# RealVeritas AI - Image Deepfake Model Training
Run this notebook in Google Colab to train your MobileNetV2 image model on GPU.

**Instructions:**
1. Go to **Runtime > Change runtime type** and ensure **Hardware accelerator** is set to **T4 GPU**.
2. Upload `archive (1).zip` and your existing `image_detector_model.h5` to the root of your Google Drive.
3. Run the cells below sequentially.

In [ ]:
from google.colab import drive
import os
import zipfile

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Path to your uploaded zip file
zip_path = '/content/drive/MyDrive/archive (1).zip'
extract_path = '/content/dataset'

# 3. Extract to Colab local storage (much faster than reading from Drive directly)
if not os.path.exists(extract_path):
    print("Extracting dataset (this will take a few minutes)...")
    os.makedirs(extract_path)
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_path)
        print("Extraction complete!")
    except Exception as e:
        print(f"Error extracting zip: {e}")
else:
    print("Dataset already extracted.")

In [ ]:
import pandas as pd
import numpy as np
import glob
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Input
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight

# 4. Locate the frames folder dynamically
try:
    frames_dir = glob.glob('/content/dataset/**/FF++C32-Frames', recursive=True)[0]
except IndexError:
    frames_dir = '/content/dataset/FF++C32-Frames' # fallback

print(f"Found frames directory at: {frames_dir}")

# 5. Build DataFrame for 3 Categories (Real, AI_Generated, AI_Manipulated)
data = []
for folder in os.listdir(frames_dir):
    folder_path = os.path.join(frames_dir, folder)
    if os.path.isdir(folder_path):
        folder_lower = folder.lower()
        
        # Map the folders from archive (1).zip to the 3 required classes
        if folder_lower == 'original':
            label = 'Real'
        elif folder_lower in ['deepfakes', 'faceshifter']:
            label = 'AI_Generated'
        else: 
            # Face2Face, FaceSwap, NeuralTextures
            label = 'AI_Manipulated'
            
        for img in os.listdir(folder_path):
            if img.lower().endswith(('.png', '.jpg', '.jpeg')):
                data.append({'file_name': os.path.join(folder_path, img), 'label_str': label})

df = pd.DataFrame(data)
print("\nClass Distribution in Dataset:")
print(df["label_str"].value_counts())

# 6. Data Generators
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    brightness_range=[0.8, 1.2],
    horizontal_flip=True,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=df,
    x_col="file_name",
    y_col="label_str",
    target_size=(128, 128),
    batch_size=32,
    class_mode="categorical",
    subset="training",
    shuffle=True
)

val_generator = train_datagen.flow_from_dataframe(
    dataframe=df,
    x_col="file_name",
    y_col="label_str",
    target_size=(128, 128),
    batch_size=32,
    class_mode="categorical",
    subset="validation",
    shuffle=False
)


In [ ]:
# 7. Class Weights
classes = np.unique(df["label_str"])
weights = compute_class_weight(class_weight="balanced", classes=classes, y=df["label_str"])
class_weight_dict = dict(zip(range(len(classes)), weights))
print("Computed Class Weights:", class_weight_dict)

# ==========================================
# 8. MODEL SETUP (INCREMENTAL LEARNING)
# ==========================================
CONTINUE_TRAINING = True
existing_model_path = '/content/drive/MyDrive/image_detector_model.h5'

model_loaded_successfully = False
if CONTINUE_TRAINING and os.path.exists(existing_model_path):
    print("\n--- LOADING EXISTING MODEL TO MAKE IT MORE POWERFUL ---")
    try:
        # Added compile=False. This fixes 99% of Google Colab model loading errors
        # when moving models between different TensorFlow versions.
        model = load_model(existing_model_path, compile=False)
        
        if model.output_shape[-1] != len(classes):
            print(f"\n[!] Adapting model output from {model.output_shape[-1]} classes to {len(classes)} classes...")
            x = model.layers[-2].output
            outputs = Dense(len(classes), activation="softmax", name="adapted_output")(x)
            model = Model(model.input, outputs)
        
        model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5),
                      loss="categorical_crossentropy", metrics=["accuracy"])
        model_loaded_successfully = True
        print("\n[+] Model loaded and adapted successfully!")
    except Exception as e:
        print(f"\n[!] ERROR loading your existing model: {e}")
        print("[!] The file might be corrupted or incompatible with Colab's TensorFlow version.")
        print("\n--- FALLING BACK TO CREATING A BRAND NEW MODEL ---")
        model_loaded_successfully = False

if not model_loaded_successfully:
    print("\nCreating a brand new MobileNetV2 model from scratch...")
    base_model = MobileNetV2(input_shape=(128, 128, 3), include_top=False, weights="imagenet")
    base_model.trainable = True
    for layer in base_model.layers[:-30]:
        layer.trainable = False
    
    inputs = Input(shape=(128, 128, 3))
    x = base_model(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation="relu")(x)
    x = Dropout(0.4)(x)
    outputs = Dense(len(classes), activation="softmax")(x)
    
    model = Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
                  loss="categorical_crossentropy", metrics=["accuracy"])

model.summary()

# 9. Checkpoints and Callbacks
checkpoint_dir = '/content/drive/MyDrive/Model_Checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

checkpoint_path = os.path.join(checkpoint_dir, 'model_epoch_{epoch:02d}_acc_{val_accuracy:.2f}.h5')
latest_model_path = '/content/drive/MyDrive/image_detector_model_latest.h5'

callbacks = [
    EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True),
    ModelCheckpoint(checkpoint_path, monitor="val_accuracy", save_best_only=False, verbose=1),
    ModelCheckpoint(latest_model_path, monitor="val_accuracy", save_best_only=False, verbose=0),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1)
]

print("\nStarting Training on GPU... (Progress bar and ETA will display below)")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

print("\nTraining complete! Final model saved to Google Drive.")